# Lakeflow Jobs: Orchestrate Your Pipelines

Welcome to the **Lakeflow Jobs** demo! This notebook teaches you how to create, configure, and manage jobs to orchestrate your Lakeflow Spark Declarative Pipelines.

---

## What you'll learn:

* **Part 1:** Understanding Lakeflow Jobs concepts
* **Part 2:** Creating jobs programmatically with the SDK and CLI
* **Part 3:** Job configuration and parameters
* **Part 4:** Scheduling and triggers
* **Part 5:** Monitoring and notifications
* **Part 6:** Best practices and patterns

---

## Prerequisites:

* Familiarity with [Lakeflow Spark Declarative Pipelines (SDP)](https://docs.databricks.com/aws/en/ldp/concepts/) concepts
* Familiarity with the [Lakeflow Pipelines Editor](https://docs.databricks.com/aws/en/ldp/multi-file-editor/) for developing pipelines
* Understanding of streaming tables, materialized views, and flows
* Basic understanding of Python

---

**Let's get started!**


## Part 1: Understanding Lakeflow Jobs 

Before creating jobs, let's understand what they are and why they matter.

---

### What are Lakeflow Jobs?

**Definition:**
* Lakeflow Jobs is **workflow automation for Databricks**, providing orchestration for data processing workloads
* Coordinate and run multiple tasks as part of a larger workflow
* Optimize and schedule the execution of frequent, repeatable tasks
* Manage complex workflows with conditional logic and dependencies
* Part of the Databricks platform (accessible via **Jobs & Pipelines** in the sidebar)

---

### **Three Core Orchestration Concepts:**

**1. Job** - The primary resource for coordinating, scheduling, and running your operations. Jobs can vary in complexity from a single task running a notebook to hundreds of tasks with conditional logic and dependencies. The tasks in a job are visually represented by a **Directed Acyclic Graph (DAG)**.

**2. Task** - A specific unit of work within a job. Each task can perform a variety of operations:
* A **pipeline task** runs a Lakeflow Spark Declarative Pipeline
* A **notebook task** runs a Databricks notebook
* A **Python script task** runs a Python file
* A **SQL task** runs SQL queries
* **Control flow tasks** add branching (`If/else`) and looping (`For each`)

**3. Trigger** - A mechanism that initiates running a job based on specific conditions or events:
* **Time-based** (cron schedule, e.g., every day at 2 AM)
* **Event-based** (e.g., when new data arrives in cloud storage)
* **Continuous** (always running)
* **Manual** (on-demand via UI or API)

---

### **Key Benefits:**

* **Automation** - Schedule pipelines to run automatically
* **Orchestration** - Chain multiple tasks with dependencies, branching, and looping
* **Monitoring** - Built-in observability via UI and `system.lakeflow` system tables
* **Notifications** - Alert on success, failure, or duration thresholds via email, Slack, or webhooks
* **Serverless** - Default serverless compute with automatic scaling and optimization
* **Control Flow** - `If/else` branching and `For each` looping for complex workflows

### Jobs vs Pipelines: What's the Difference?

**Understanding the relationship:**

---

### **Lakeflow Spark Declarative Pipeline (SDP):**

**What it is:**
* The **data transformation logic**
* Defines streaming tables, materialized views, and flows
* Written in Python (using `pyspark.pipelines`) or SQL source files
* Developed in the **Lakeflow Pipelines Editor** (multi-file IDE)
* Processes data from source to target using declarative definitions

**Example (Python):**
```python
from pyspark import pipelines as dp

@dp.table
def bronze_customers():
    return spark.readStream.table("raw.customers")

@dp.materialized_view
def silver_customers():
    return spark.read.table("bronze_customers").filter(...)
```

**Example (SQL):**
```sql
CREATE OR REFRESH STREAMING TABLE bronze_customers
AS SELECT * FROM STREAM(raw.customers);

CREATE OR REFRESH MATERIALIZED VIEW silver_customers
AS SELECT * FROM bronze_customers WHERE ...;
```

**Think of it as:** The "what" — what data transformations to perform

---

### **Lakeflow Job:**

**What it is:**
* The **orchestration wrapper** around pipelines and other tasks
* Defines **when** and **how** to run pipelines
* Manages scheduling, triggers, and monitoring
* Can run multiple tasks (pipelines, notebooks, Python scripts, SQL, etc.)
* Supports control flow with `If/else` branching and `For each` looping

**Think of it as:** The "when" and "how" — when to run and how to orchestrate

---

### **Relationship:**

```
┌─────────────────────────────────────────┐
│           Lakeflow Job                  │
│    (Orchestration & Scheduling)         │
│                                         │
│  ┌───────────────────────────────────┐ │
│  │  Task 1: SDP Pipeline             │ │
│  │  (Streaming tables, MVs, flows)   │ │
│  └───────────────────────────────────┘ │
│               │                         │
│  ┌───────────────────────────────────┐ │
│  │  Task 2: Notebook                 │ │
│  │  (Post-processing / reporting)    │ │
│  └───────────────────────────────────┘ │
│               │                         │
│  ┌───────────────────────────────────┐ │
│  │  Task 3: If/else branching        │ │
│  │  (Conditional logic)              │ │
│  └───────────────────────────────────┘ │
└─────────────────────────────────────────┘
```

**Key point:** Jobs **orchestrate** pipelines; pipelines **transform** data

---

### **Important: Triggered vs Continuous Pipelines**

* **Triggered pipelines** stop after refreshing all tables — ideal for job tasks
* **Continuous pipelines** run continuously and process data as it arrives — **cannot** be added as a job task (they don't need external triggering)
* Only **triggered pipelines** can be scheduled as pipeline tasks in a job

### Job Components

A Lakeflow Job consists of several key components:

---

### **1. Tasks**

**What they are:**
* Individual units of work within a job
* Execute in sequence, parallel, or conditionally based on dependencies
* Visually represented as a Directed Acyclic Graph (DAG)

**Standard task types:**
* **Pipeline Task** — Run a Lakeflow Spark Declarative Pipeline (triggered mode only)
* **Notebook Task** — Run a Databricks notebook
* **Python Script Task** — Run a Python file
* **SQL Task** — Run SQL queries
* **dbt Task** — Run dbt models
* **JAR Task** — Run a JAR file

**Control flow task types:**
* **If/else Task** — Conditional branching based on task values, job parameters, or dynamic values
* **For each Task** — Loop over a set of parameters, running a nested task for each iteration
* **Run if Task** — Conditionally run based on upstream task outcomes (all succeeded, at least one failed, etc.)

---

### **2. Triggers**

**What they are:**
* Mechanisms that initiate running a job
* Define **when** the job runs automatically

**Trigger types:**
* **Cron schedule** — Time-based (e.g., daily at 2 AM)
* **File arrival** — Event-driven, when files arrive in a Unity Catalog storage location
* **Continuous** — Always running, restarts after completion
* **Manual** — On-demand via UI, API, or CLI

---

### **3. Compute Configuration**

**Options:**
* **Serverless** — Default for new jobs. No cluster management, auto-scaling, Photon-enabled
* **Job cluster** — Dedicated cluster created per run, terminated after completion
* **Existing cluster** — Use a running cluster (dev/testing only, not recommended for production)

---

### **4. Parameters**

**What they are:**
* Runtime values that are automatically pushed to tasks within the job
* Allow reusable jobs with different inputs
* Support dynamic value references (e.g., `{{job.start_time.iso_date}}`)

---

### **5. Notifications**

**What they are:**
* Alerts sent on job events (start, success, failure, duration threshold exceeded)
* Channels: email, Slack, custom webhooks, PagerDuty, and more

---

### **6. Git Integration**

**What it is:**
* Source control settings for job tasks
* Tasks can reference notebooks/files from a Git repo
* Enables version-controlled production deployments

%undefined
## Part 2: Creating Jobs Programmatically 🔧

Let's learn how to create and manage jobs using the Databricks Jobs API.

---

### Managing Jobs Programmatically

**Databricks provides multiple tools for automating job creation and management:**

---

### **1. Databricks SDK (Recommended for notebooks)**
```python
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.jobs import JobSettings as Job

w = WorkspaceClient()
j = w.jobs.create(**Job.from_dict({
    "name": "my_pipeline_job",
    "tasks": [{...}]
}).as_shallow_dict())
```

### **2. Databricks CLI**
```bash
databricks jobs create --json @job-config.json
databricks jobs run-now --job-id 12345
```

### **3. Declarative Automation Bundles (Recommended for CI/CD)**
```yaml
# databricks.yml
resources:
  jobs:
    my_pipeline_job:
      name: my_pipeline_job
      tasks:
        - task_key: run_pipeline
          pipeline_task:
            pipeline_id: abc-123-def
```
* Formerly known as Databricks Asset Bundles
* Define jobs as YAML source files alongside your code
* Deploy via `databricks bundle deploy`
* Supports environments (dev/staging/prod) and CI/CD integration

### **4. REST API**
```python
import requests

response = requests.post(
    f"{workspace_url}/api/2.1/jobs/create",
    headers={"Authorization": f"Bearer {token}"},
    json={"name": "my_pipeline_job", "tasks": [...]}
)
```

---

### **Why Automate?**

* **Infrastructure as Code** — Version control your job definitions
* **CI/CD** — Deploy jobs from automated pipelines with Declarative Automation Bundles
* **Consistency** — Standardize job configurations across environments
* **Scalability** — Manage hundreds of jobs programmatically

---

**In this demo, we'll use the Databricks SDK from within a notebook.**

In [0]:
# Install Databricks SDK (latest version)
%pip install --upgrade databricks-sdk==0.74.0 --quiet

dbutils.library.restartPython()

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service import jobs
from databricks.sdk.service.jobs import JobSettings as Job
import json

# Initialize the Databricks workspace client
# This automatically uses the notebook's authentication context
w = WorkspaceClient()

print("Databricks SDK initialized successfully!")
print(f"Workspace: {w.config.host}")

### Creating Your First Job

Let's create a simple job that runs a Lakeflow Spark Declarative Pipeline.

**Steps:**
1. Define the job configuration
2. Create the job using the SDK
3. Verify the job was created

**Note:** Only **triggered** pipelines can be added as pipeline tasks in a job. Continuous pipelines run independently and don't need external triggering.

---

In [0]:
# ============================================
# EXAMPLE 1: Simple Pipeline Job
# ============================================

# Create a job that runs a Lakeflow Spark Declarative Pipeline.
# Only triggered pipelines can be used as job tasks.

try:
    # List existing pipelines
    pipelines = w.pipelines.list_pipelines()
    pipeline_list = list(pipelines)
    
    if pipeline_list:
        # Use the first pipeline as an example
        example_pipeline = pipeline_list[0]
        pipeline_id = example_pipeline.pipeline_id
        pipeline_name = example_pipeline.name
        
        print(f"Found pipeline: {pipeline_name}")
        print(f"Pipeline ID: {pipeline_id}")
        
        # Create a simple job with a pipeline task
        # Serverless compute is used by default for supported task types
        job = w.jobs.create(
            name=f"Demo: {pipeline_name} - Daily Run",
            tasks=[
                jobs.Task(
                    task_key="run_pipeline",
                    description="Run the SDP pipeline (incremental refresh)",
                    pipeline_task=jobs.PipelineTask(
                        pipeline_id=pipeline_id,
                        full_refresh=False  # Incremental run (default)
                    )
                )
            ],
            # Tags help organize, filter, and track costs
            tags={
                "environment": "demo",
                "created_by": "lakeflow_jobs_demo"
            }
        )
        
        print(f"\nJob created successfully!")
        print(f"Job ID: {job.job_id}")
        print(f"View job: {w.config.host}/#job/{job.job_id}")
        
    else:
        print("No pipelines found. Please create a pipeline first.")
        print("Tip: Use the Lakeflow Pipelines Editor to create a new pipeline.")
        
except Exception as e:
    print(f"Error: {e}")
    print("Make sure you have permissions to create jobs.")

### Understanding the Job Creation Code

**What we did:**

---

### **1. Initialized the Workspace Client:**
```python
w = WorkspaceClient()
```
* Connects to your Databricks workspace
* Uses notebook authentication automatically
* Provides access to all Databricks APIs (Jobs, Pipelines, Clusters, etc.)

---

### **2. Created a Job:**
```python
job = w.jobs.create(
    name="Demo: Pipeline - Daily Run",
    tasks=[...]
)
```
* `name` — Human-readable job name
* `tasks` — List of tasks to execute (DAG)

---

### **3. Defined a Pipeline Task:**
```python
jobs.Task(
    task_key="run_pipeline",              # Unique identifier for DAG dependencies
    description="Run the SDP pipeline",   # Human-readable description
    pipeline_task=jobs.PipelineTask(
        pipeline_id=pipeline_id,          # Which pipeline to run
        full_refresh=False                # Incremental vs full refresh
    )
)
```

**Key parameters:**
* `task_key` — Unique identifier used for task dependencies
* `pipeline_task` — Specifies this is a Lakeflow SDP pipeline task
* `pipeline_id` — The ID of the triggered pipeline to run
* `full_refresh` — `False` for incremental (process only new data), `True` to reprocess all data

**Note:** The pipeline manages its own compute configuration (SDP handles cluster creation). Serverless compute is the default for other task types like notebook and Python script tasks.

---

### **4. Added Tags:**
```python
tags={
    "environment": "demo",
    "created_by": "lakeflow_jobs_demo"
}
```
* Helps organize, filter, and search jobs
* Useful for cost tracking via `system.billing.usage` system tables
* Supports governance and ownership tracking

%undefined
## Part 3: Job Configuration and Parameters ⚙️

Let's explore advanced job configurations including compute, parameters, and task dependencies.

---

### Compute Configuration Options

**Jobs can use different compute strategies depending on the task type:**

---

### **1. Serverless Compute (Default and Recommended)**

**What it is:**
* Managed compute that scales automatically with Photon enabled
* No cluster management, configuration, or permissions required
* Databricks automatically optimizes instance types, memory, and processing
* Runtime version is automatically upgraded

**Supported task types:** Notebook, Python script, dbt, Python wheel, JAR

**When to use:**
* Most production workloads
* When you want simplicity and cost efficiency
* Variable workload sizes

**Configuration:**
```python
# Serverless is the default — no additional compute config needed!
jobs.Task(
    task_key="my_notebook_task",
    notebook_task=jobs.NotebookTask(
        notebook_path="/path/to/notebook"
    )
    # Serverless compute is automatically selected
)
```

---

### **2. Pipeline Compute (Managed by SDP)**

**What it is:**
* For pipeline tasks, compute is managed by the pipeline definition itself
* Configured in the pipeline settings, not in the job
* SDP handles cluster creation, scaling, and termination

**Configuration:**
```python
# Pipeline tasks use their own compute — no cluster config in the job
jobs.Task(
    task_key="run_pipeline",
    pipeline_task=jobs.PipelineTask(
        pipeline_id=pipeline_id
    )
)
```

---

### **3. Job Cluster**

**What it is:**
* Dedicated cluster created for the job run
* Terminated after the job completes
* Full control over cluster configuration

**When to use:**
* Need specific cluster configurations
* Custom libraries or init scripts
* Specific instance types required

---

### **4. Existing Cluster**

**What it is:**
* Use an already-running cluster
* Cluster must be running when the job starts

**When to use:**
* Development and testing only
* Not recommended for production (cluster may be terminated or shared)

In [0]:
# ============================================
# EXAMPLE 2: Job with Parameters
# ============================================

# Create a job that accepts parameters.
# Parameters are runtime values automatically pushed to tasks.
# You can use dynamic value references like {{job.start_time.iso_date}}

try:
    pipelines = w.pipelines.list_pipelines()
    pipeline_list = list(pipelines)
    
    if pipeline_list:
        pipeline_id = pipeline_list[0].pipeline_id
        
        # Create job with parameters
        job_with_params = w.jobs.create(
            name="Demo: Parameterized Pipeline Job",
            tasks=[
                jobs.Task(
                    task_key="run_pipeline_with_params",
                    description="Run pipeline with custom parameters",
                    pipeline_task=jobs.PipelineTask(
                        pipeline_id=pipeline_id,
                        full_refresh=False
                    )
                )
            ],
            # Define default parameters
            # These can be overridden at runtime via UI, API, or CLI
            parameters=[
                jobs.JobParameterDefinition(
                    name="environment",
                    default="development"
                ),
                jobs.JobParameterDefinition(
                    name="start_date",
                    default="2026-01-01"
                ),
                jobs.JobParameterDefinition(
                    name="run_date",
                    default="{{job.start_time.iso_date}}"  # Dynamic value reference
                )
            ],
            tags={
                "environment": "demo",
                "type": "parameterized"
            }
        )
        
        print(f"Parameterized job created!")
        print(f"Job ID: {job_with_params.job_id}")
        print(f"\nParameters:")
        for param in job_with_params.settings.parameters:
            print(f"   - {param.name}: {param.default}")
        print(f"\nView job: {w.config.host}/#job/{job_with_params.job_id}")
        
    else:
        print("No pipelines found.")
        
except Exception as e:
    print(f"Error: {e}")

In [0]:
# ============================================
# EXAMPLE 3: Multi-Task Job with Dependencies
# ============================================

# Create a job with multiple tasks that form a DAG.
# Tasks can run in sequence (via depends_on) or in parallel.
# Notebook tasks use serverless compute by default.

try:
    pipelines = w.pipelines.list_pipelines()
    pipeline_list = list(pipelines)
    
    if pipeline_list:
        pipeline_id = pipeline_list[0].pipeline_id
        notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        
        # Create a multi-task job
        multi_task_job = w.jobs.create(
            name="Demo: Multi-Task Pipeline Job",
            tasks=[
                # Task 1: Run the SDP pipeline
                jobs.Task(
                    task_key="ingest_and_transform",
                    description="Run SDP pipeline to ingest and transform data",
                    pipeline_task=jobs.PipelineTask(
                        pipeline_id=pipeline_id,
                        full_refresh=False
                    )
                ),
                # Task 2: Run a notebook (depends on Task 1)
                # Uses serverless compute by default
                jobs.Task(
                    task_key="generate_report",
                    description="Generate summary report after pipeline completes",
                    depends_on=[
                        jobs.TaskDependency(task_key="ingest_and_transform")
                    ],
                    notebook_task=jobs.NotebookTask(
                        notebook_path=notebook_path,
                        base_parameters={
                            "report_type": "summary",
                            "output_format": "html"
                        }
                    )
                    # No compute config needed — serverless is the default
                ),
                # Task 3: Send notification (depends on Task 2)
                jobs.Task(
                    task_key="send_notification",
                    description="Send completion notification",
                    depends_on=[
                        jobs.TaskDependency(task_key="generate_report")
                    ],
                    notebook_task=jobs.NotebookTask(
                        notebook_path=notebook_path,
                        base_parameters={
                            "action": "notify",
                            "message": "Pipeline completed successfully"
                        }
                    )
                )
            ],
            tags={
                "environment": "demo",
                "type": "multi_task"
            }
        )
        
        print(f"Multi-task job created!")
        print(f"Job ID: {multi_task_job.job_id}")
        print(f"\nTask DAG (sequential):")
        print(f"   1. ingest_and_transform (SDP Pipeline)")
        print(f"   2. generate_report (Notebook) -> depends on task 1")
        print(f"   3. send_notification (Notebook) -> depends on task 2")
        print(f"\nView job: {w.config.host}/#job/{multi_task_job.job_id}")
        
    else:
        print("No pipelines found.")
        
except Exception as e:
    print(f"Error: {e}")

### Task Dependencies and Control Flow

**How task dependencies and control flow work:**

---

### **Sequential Execution:**

```python
tasks=[
    jobs.Task(task_key="task_1", pipeline_task=...),
    jobs.Task(task_key="task_2",
        depends_on=[jobs.TaskDependency(task_key="task_1")],
        notebook_task=...),
    jobs.Task(task_key="task_3",
        depends_on=[jobs.TaskDependency(task_key="task_2")],
        notebook_task=...)
]
```
```
task_1 → task_2 → task_3
```

---

### **Parallel Execution:**

Tasks with the same parent dependency run in parallel:

```
        task_1
       /      \
   task_2a  task_2b  (run in parallel)
       \      /
        task_3
```

---

### **Control Flow: If/else Branching**

Use `If/else` tasks to add conditional logic based on:
* **Task values** — Values returned by previous tasks via `taskValues`
* **Job parameters** — Runtime parameters defined on the job
* **Dynamic values** — References like `{{job.start_time.iso_date}}`

Supported operators: `==`, `!=`, `>`, `>=`, `<`, `<=`

---

### **Control Flow: For each Looping**

Use `For each` tasks to run a nested task in a loop:
* Define an input array (JSON) of values to iterate over
* Each iteration runs the nested task with different parameters
* Set concurrency to control parallel iterations
* Reference parameters with `{{input}}` or `{{input.<key>}}`

---

### **Run if Conditions:**

Configure downstream tasks to run based on upstream task outcomes:
* **All succeeded** — Run only if all dependencies succeeded
* **At least one succeeded** — Run if any dependency succeeded
* **None failed** — Run if no dependencies failed (includes skipped)
* **All done** — Run regardless of dependency outcomes
* **At least one failed** / **All failed** — For error handling tasks

---

### **Key Points:**

* Tasks without dependencies run immediately
* Tasks with dependencies wait for parent tasks to complete
* Multiple tasks can depend on the same parent (parallel execution)
* A task can depend on multiple parents (fan-in pattern)
* Control flow tasks enable complex conditional and iterative workflows

%undefined
## Part 4: Scheduling and Triggers ⏰

Let's learn how to schedule jobs to run automatically.

---

### Job Triggers and Scheduling

**Lakeflow Jobs support multiple trigger methods to automatically initiate runs:**

---

### **1. Time-Based Schedule (Cron)**

**What it is:**
* Schedule-based triggering using Quartz cron expressions
* Most common trigger method for batch processing
* Supports complex schedules

**Cron format (Quartz — 7 fields):**
```
┌─────────────── second (0-59)
│ ┌───────────── minute (0-59)
│ │ ┌─────────── hour (0-23)
│ │ │ ┌───────── day of month (1-31)
│ │ │ │ ┌─────── month (1-12 or JAN-DEC)
│ │ │ │ │ ┌───── day of week (1-7 or SUN-SAT)
│ │ │ │ │ │ ┌─── year (optional)
* * * * * * *
```

**Common examples:**
```python
# Every day at midnight
"0 0 0 * * ?"

# Every hour
"0 0 * * * ?"

# Every 15 minutes
"0 */15 * * * ?"

# Every Monday at 9 AM
"0 0 9 ? * MON"

# Every weekday at 6 AM
"0 0 6 ? * MON-FRI"

# First day of every month at midnight
"0 0 0 1 * ?"
```

---

### **2. File Arrival Trigger**

**What it is:**
* Event-driven trigger that runs when new files arrive in a **Unity Catalog storage location**
* Monitors cloud storage paths for new data
* Ideal for unpredictable data arrival patterns

**When to use:**
* Process files as they arrive
* Event-driven ETL workflows
* Data arrives on an irregular schedule

---

### **3. Continuous Trigger**

**What it is:**
* Job runs continuously, restarting automatically after completion
* Useful for near-real-time processing (not streaming — that's what continuous pipelines are for)

**When to use:**
* Batch processing that should re-run as soon as the previous run completes
* Near-real-time requirements with batch semantics

---

### **4. Manual (On-Demand)**

**What it is:**
* No automatic trigger configured
* Run via UI (**Run now** button), API, CLI, or external orchestration (e.g., Apache Airflow)

**When to use:**
* Ad-hoc processing
* Testing and development
* Triggered by external systems or CI/CD pipelines

In [0]:
# ============================================
# EXAMPLE 4: Scheduled Job (Daily at Midnight)
# ============================================

# Create a job with a time-based trigger (cron schedule).
# The job will automatically run at the specified time.

try:
    pipelines = w.pipelines.list_pipelines()
    pipeline_list = list(pipelines)
    
    if pipeline_list:
        pipeline_id = pipeline_list[0].pipeline_id
        
        # Create a scheduled job
        scheduled_job = w.jobs.create(
            name="Demo: Daily Pipeline Job (Midnight)",
            tasks=[
                jobs.Task(
                    task_key="daily_pipeline_run",
                    description="Run pipeline daily at midnight",
                    pipeline_task=jobs.PipelineTask(
                        pipeline_id=pipeline_id,
                        full_refresh=False
                    )
                )
            ],
            # Time-based trigger using Quartz cron expression
            schedule=jobs.CronSchedule(
                quartz_cron_expression="0 0 0 * * ?",
                timezone_id="UTC",
                pause_status=jobs.PauseStatus.UNPAUSED
            ),
            tags={
                "environment": "demo",
                "schedule": "daily"
            }
        )
        
        print(f"Scheduled job created!")
        print(f"Job ID: {scheduled_job.job_id}")
        print(f"Schedule: Daily at midnight (UTC)")
        print(f"Cron: {scheduled_job.settings.schedule.quartz_cron_expression}")
        print(f"View job: {w.config.host}/#job/{scheduled_job.job_id}")
        
    else:
        print("No pipelines found.")
        
except Exception as e:
    print(f"Error: {e}")

In [0]:
# ============================================
# EXAMPLE 5: File Arrival Trigger
# ============================================

# File arrival triggers run a job when new files arrive
# in a Unity Catalog storage location.
# Requires appropriate permissions on the storage location.

try:
    pipelines = w.pipelines.list_pipelines()
    pipeline_list = list(pipelines)
    
    if pipeline_list:
        pipeline_id = pipeline_list[0].pipeline_id
        
        # Create a file-triggered job
        file_trigger_job = w.jobs.create(
            name="Demo: File Arrival Triggered Job",
            tasks=[
                jobs.Task(
                    task_key="process_new_files",
                    description="Process pipeline when new files arrive",
                    pipeline_task=jobs.PipelineTask(
                        pipeline_id=pipeline_id,
                        full_refresh=False
                    )
                )
            ],
            # Trigger on file arrival in a Unity Catalog storage location
            trigger=jobs.TriggerSettings(
                file_arrival=jobs.FileArrivalTriggerConfiguration(
                    url="s3://my-bucket/incoming-data/",  # Change to your UC storage location
                    min_time_between_triggers_seconds=60,  # Min 60s between triggers
                    wait_after_last_change_seconds=30      # Wait 30s after last file change
                )
            ),
            tags={
                "environment": "demo",
                "trigger": "file_arrival"
            }
        )
        
        print(f"File-triggered job created!")
        print(f"Job ID: {file_trigger_job.job_id}")
        print(f"Monitoring: s3://my-bucket/incoming-data/")
        print(f"Min time between triggers: 60 seconds")
        print(f"View job: {w.config.host}/#job/{file_trigger_job.job_id}")
        print(f"\nNote: Update the storage path to your actual Unity Catalog location")
        
    else:
        print("No pipelines found.")
        
except Exception as e:
    print(f"Error: {e}")
    print(f"File arrival triggers require Unity Catalog storage permissions")

### Challenge 1: Create a Scheduled Job

**Your task:**

Create a job that runs a pipeline every Monday at 9 AM.

---

**Requirements:**

1. **Job name:** "Challenge 1: Weekly Pipeline Job"
2. **Task:**
   * Task key: "weekly_pipeline_run"
   * Use any existing triggered pipeline
   * Incremental refresh (not full refresh)
3. **Trigger (cron schedule):**
   * Every Monday at 9:00 AM UTC
   * Cron expression: `"0 0 9 ? * MON"`
   * Timezone: "UTC"
   * Status: UNPAUSED
4. **Tags:**
   * environment: "challenge"
   * schedule: "weekly"

---

**Write your code below:**

In [0]:
# ============================================
# CHALLENGE 1: Your solution here
# ============================================

# TODO: Create a job that runs every Monday at 9 AM



%undefined
## Part 5: Monitoring and Notifications 📊

Let's learn how to monitor job execution and set up notifications.

---

### Monitoring Job Execution

**Multiple ways to monitor Lakeflow Jobs:**

---

### **1. Databricks UI (Jobs & Pipelines)**

* View all jobs and their status from the **Jobs & Pipelines** page in the sidebar
* See run history, duration, and task-level details
* Inspect the DAG graph view with task-level execution insights
* Access logs, error messages, and metrics
* Filter by job properties, tags, and ownership

---

### **2. Jobs API / SDK**

**List job runs:**
```python
runs = w.jobs.list_runs(job_id=job_id, limit=10)

for run in runs:
    print(f"Run {run.run_id}: {run.state.life_cycle_state}")
```

**Get run details:**
```python
run = w.jobs.get_run(run_id=run_id)

print(f"Status: {run.state.life_cycle_state}")
print(f"Result: {run.state.result_state}")
print(f"Start: {run.start_time}")
print(f"End: {run.end_time}")
```

---

### **3. System Tables (`system.lakeflow` schema)**

Databricks provides system tables for querying job runs and tasks across your account:

* `system.lakeflow.jobs` — Job definitions and metadata
* `system.lakeflow.job_run_timeline` — Run history with timing and status

Join with `system.billing.usage` to monitor costs:

```sql
SELECT
    jobs.name,
    usage.usage_metadata.job_id,
    SUM(usage.usage_quantity * list_prices.pricing.default) as cost
FROM system.billing.usage
LEFT JOIN system.lakeflow.jobs USING (workspace_id, job_id)
INNER JOIN system.billing.list_prices ON ...
WHERE billing_origin_product = 'JOBS'
GROUP BY ALL
```

---

### **4. Run States**

**Life cycle states:**
* `PENDING` — Waiting to start
* `RUNNING` — Currently executing
* `TERMINATING` — Shutting down
* `TERMINATED` — Completed
* `SKIPPED` — Skipped due to conditions
* `INTERNAL_ERROR` — System error

**Result states (for terminated runs):**
* `SUCCESS` — Completed successfully
* `FAILED` — Failed with errors
* `TIMEDOUT` — Exceeded timeout
* `CANCELED` — Manually canceled

---

### **5. Key Metrics to Monitor**

* Success / failure rate
* Run duration trends
* Cost per run (via system tables)
* Data quality metrics
* Pipeline backlog (for streaming)

In [0]:
# ============================================
# EXAMPLE 6: List and Monitor Jobs
# ============================================

try:
    # List all jobs (limit to 10 for demo)
    all_jobs = w.jobs.list(limit=10)
    
    print("Your Jobs:\n")
    
    for job in all_jobs:
        print(f"Job ID: {job.job_id}")
        print(f"Name: {job.settings.name}")
        
        # Get recent runs for this job
        runs = list(w.jobs.list_runs(job_id=job.job_id, limit=3))
        
        if runs:
            print(f"Recent Runs:")
            for run in runs:
                status = run.state.life_cycle_state
                result = run.state.result_state if run.state.result_state else "N/A"
                print(f"   - Run {run.run_id}: {status} - {result}")
        else:
            print(f"No runs yet")
        
        print(f"View: {w.config.host}/#job/{job.job_id}")
        print("-" * 60)
        
except Exception as e:
    print(f"Error: {e}")

### Notification Configuration

**Set up alerts for job events via email, Slack, webhooks, and more:**

---

### **Email Notifications**

```python
email_notifications=jobs.JobEmailNotifications(
    on_start=["team@company.com"],
    on_success=["manager@company.com"],
    on_failure=["oncall@company.com", "team@company.com"],
    on_duration_warning_threshold_exceeded=["manager@company.com"],
    no_alert_for_skipped_runs=True
)
```

**Available events:**
* `on_start` — Job starts running
* `on_success` — Job completes successfully
* `on_failure` — Job fails
* `on_duration_warning_threshold_exceeded` — Job runs longer than expected

---

### **Webhook Notifications**

Integrate with external systems (Slack, PagerDuty, Jira, custom monitoring):

```python
webhook_notifications=jobs.WebhookNotifications(
    on_start=[jobs.Webhook(id="webhook-123")],
    on_success=[jobs.Webhook(id="webhook-456")],
    on_failure=[jobs.Webhook(id="webhook-789")]
)
```

---

### **Task-Level Notifications**

You can also add notifications on individual tasks (not just the job level) for fine-grained alerting on specific pipeline stages.

---

### **Best Practices**

* Always notify on failures for production jobs
* Use different recipients for different severity levels
* Set duration thresholds to catch jobs that run unusually long
* Avoid spamming with success notifications for non-critical jobs
* Use webhooks for integration with team chat and incident management

In [0]:
# ============================================
# EXAMPLE 7: Job with Email Notifications
# ============================================

try:
    pipelines = w.pipelines.list_pipelines()
    pipeline_list = list(pipelines)
    
    if pipeline_list:
        pipeline_id = pipeline_list[0].pipeline_id
        
        # Get current user email for notifications
        current_user = w.current_user.me()
        user_email = current_user.user_name
        
        # Create job with notifications
        notified_job = w.jobs.create(
            name="Demo: Job with Notifications",
            tasks=[
                jobs.Task(
                    task_key="pipeline_with_alerts",
                    description="Pipeline with email notifications",
                    pipeline_task=jobs.PipelineTask(
                        pipeline_id=pipeline_id,
                        full_refresh=False
                    )
                )
            ],
            # Email notifications
            email_notifications=jobs.JobEmailNotifications(
                on_failure=[user_email],      # Always notify on failure
                on_success=[user_email],      # Optional success notification
                no_alert_for_skipped_runs=True
            ),
            # Job-level timeout
            timeout_seconds=3600,  # 1 hour timeout
            tags={
                "environment": "demo",
                "notifications": "enabled"
            }
        )
        
        print(f"Job with notifications created!")
        print(f"Job ID: {notified_job.job_id}")
        print(f"Notifications sent to: {user_email}")
        print(f"Timeout: 1 hour")
        print(f"View job: {w.config.host}/#job/{notified_job.job_id}")
        
    else:
        print("No pipelines found.")
        
except Exception as e:
    print(f"Error: {e}")

### Running Jobs Programmatically

**Trigger job runs via SDK, CLI, or API:**

---

### **Run Now (Manual Trigger)**

```python
# Run a job immediately
run = w.jobs.run_now(job_id=job_id)
print(f"Started run: {run.run_id}")
```

**Run with custom parameters:**
```python
# Override default parameters at runtime
run = w.jobs.run_now(
    job_id=job_id,
    job_parameters={
        "environment": "production",
        "start_date": "2026-01-26"
    }
)
```

**Via CLI:**
```bash
databricks jobs run-now --job-id 12345
```

---

### **Wait for Completion**

```python
# Start run and block until completion
run = w.jobs.run_now(job_id=job_id)
result = w.jobs.wait_get_run_job_terminated_or_skipped(run_id=run.run_id)

if result.state.result_state == jobs.RunResultState.SUCCESS:
    print("Job completed successfully!")
else:
    print(f"Job failed: {result.state.state_message}")
```

---

### **Cancel a Run**

```python
w.jobs.cancel_run(run_id=run_id)
print(f"Canceled run: {run_id}")
```

In [0]:
# ============================================
# EXAMPLE 8: Run a Job and Monitor
# ============================================

try:
    # Get the first demo job we created
    all_jobs = list(w.jobs.list(limit=5))
    
    if all_jobs:
        # Find a job created by this demo
        demo_job = None
        for job in all_jobs:
            if "Demo:" in job.settings.name:
                demo_job = job
                break
        
        if demo_job:
            print(f"Running job: {demo_job.settings.name}")
            print(f"Job ID: {demo_job.job_id}")
            
            # Start the job
            run = w.jobs.run_now(job_id=demo_job.job_id)
            
            print(f"\nStarted run: {run.run_id}")
            print(f"View run: {w.config.host}/#job/{demo_job.job_id}/run/{run.run_id}")
            print(f"\nJob is running... (not waiting for completion in demo)")
            print(f"Check the link above to monitor progress")
            
            # Note: Not waiting for completion to avoid long delays in the demo.
            # In production, use:
            # result = w.jobs.wait_get_run_job_terminated_or_skipped(run_id=run.run_id)
            
        else:
            print("No demo jobs found. Create a job first.")
    else:
        print("No jobs found.")
        
except Exception as e:
    print(f"Error: {e}")

### Challenge 2: Create a Job with Notifications

**Your task:**

Create a job with email notifications and run it.

---

**Requirements:**

1. **Job name:** "Challenge 2: Monitored Pipeline Job"
2. **Task:**
   * Task key: "monitored_pipeline"
   * Use any existing triggered pipeline
   * Incremental refresh
3. **Notifications:**
   * Send email to your user email on failure
   * Send email to your user email on success
   * No alerts for skipped runs
4. **Timeout:** 30 minutes (1800 seconds)
5. **Tags:**
   * environment: "challenge"
   * monitoring: "enabled"
6. **After creating the job, run it using `w.jobs.run_now()`**

---

**Hints:**
* Use `w.current_user.me().user_name` to get your email
* Use `jobs.JobEmailNotifications()` for notifications
* Use `timeout_seconds` parameter

---

**Write your code below:**

In [0]:
# ============================================
# CHALLENGE 2: Your solution here
# ============================================

# TODO: Create a job with notifications and run it



%undefined
## Part 6: Best Practices and Patterns ✅

Let's cover best practices for production Lakeflow Jobs.

---

### Lakeflow Jobs Best Practices

---

### **1. Naming Conventions**

**Use descriptive, consistent names:**

```python
# Good
"Production: Customer Pipeline - Daily"
"Dev: Sales Analytics - Hourly"
"Staging: Inventory Sync - Event-driven"

# Bad
"job1"
"test"
"my_pipeline"
```

**Include:** Environment, purpose, and frequency.

---

### **2. Use Tags for Organization**

```python
tags={
    "environment": "production",
    "team": "data-engineering",
    "cost_center": "analytics",
    "criticality": "high",
    "owner": "data-team@company.com"
}
```

**Benefits:** Filtering, cost allocation (via `system.billing.usage`), ownership tracking, governance.

---

### **3. Compute Strategy**

**For Pipeline Tasks:**
* Compute is managed by the pipeline definition (SDP handles it)
* Use serverless pipelines for simplicity
* Don't override pipeline compute in the job

**For Notebook / Python Script Tasks:**
* Use **serverless compute** (the default) for most workloads
* Only use job clusters when you need specific configurations
* Avoid existing clusters in production

---

### **4. Error Handling and Retries**

```python
# Set retries for transient failures
max_retries=3,
min_retry_interval_millis=60000,  # 1 minute between retries
retry_on_timeout=True,

# Set timeouts to prevent runaway jobs
timeout_seconds=7200,  # 2 hours max

# Always notify on failures
email_notifications=jobs.JobEmailNotifications(
    on_failure=["oncall@company.com"]
)
```

---

### **5. Scheduling Strategy**

**Batch processing:** Use cron schedules during off-peak hours. Consider data freshness requirements.

**Event-driven:** Use file arrival triggers with appropriate `min_time_between_triggers_seconds` to avoid over-triggering.

**Streaming:** Use continuous pipeline mode (not a job trigger) for sub-minute latency. Only triggered pipelines can be job tasks.

---

### **6. Monitoring with System Tables**

Use `system.lakeflow` schema for cross-account monitoring:

```sql
-- Failed jobs analysis (last 30 days)
SELECT j.name, COUNT(*) as runs,
       SUM(CASE WHEN t.result_state IN ('ERROR','FAILED') THEN 1 ELSE 0 END) as failures
FROM system.lakeflow.job_run_timeline t
JOIN system.lakeflow.jobs j USING (workspace_id, job_id)
WHERE t.period_end_time >= CURRENT_DATE() - INTERVAL 30 DAYS
  AND t.result_state IS NOT NULL
GROUP BY j.name
ORDER BY failures DESC
```

Join with `system.billing.usage` and `system.billing.list_prices` for cost analysis.

---

### **7. Parameters and Dynamic Values**

```python
parameters=[
    jobs.JobParameterDefinition(
        name="environment",
        default="production"
    ),
    jobs.JobParameterDefinition(
        name="run_date",
        default="{{job.start_time.iso_date}}"  # Dynamic value reference
    )
]
```

Dynamic values like `{{job.start_time.iso_date}}` are resolved at runtime.

---

### **8. CI/CD with Declarative Automation Bundles**

**Treat jobs as code using Declarative Automation Bundles (formerly Databricks Asset Bundles):**

```yaml
# databricks.yml
bundle:
  name: customer-pipeline

resources:
  jobs:
    customer_pipeline_daily:
      name: "Production: Customer Pipeline - Daily"
      tasks:
        - task_key: run_pipeline
          pipeline_task:
            pipeline_id: ${var.pipeline_id}

targets:
  dev:
    default: true
  prod:
    workspace:
      host: https://prod-workspace.databricks.com
```

**Deploy:**
```bash
databricks bundle validate
databricks bundle deploy -t prod
databricks bundle run -t prod customer_pipeline_daily
```

**Benefits:** Version control, peer review, automated deployment, environment management.

---

### **9. Testing Strategy**

1. **Development:** Create dev version with dev pipelines and data. Test manually.
2. **Staging:** Deploy with Declarative Automation Bundles to staging target. Test scheduling.
3. **Production:** Deploy to prod target with monitoring. Start with manual runs, enable schedule after validation.

---

### **10. Documentation**

```python
jobs.Task(
    task_key="ingest_customers",
    description="Ingest customer data from S3, apply SDP transformations and data quality checks. Runs daily at midnight. Owner: data-team@company.com",
    pipeline_task=...
)
```

Include: Purpose, data sources/targets, schedule, owner, and dependencies.

In [0]:
# ============================================
# EXAMPLE 9: Production-Ready Job
# ============================================

# Complete production-ready job with all best practices applied:
# - SDP pipeline task with incremental refresh
# - Cron schedule (daily at 2 AM UTC, off-peak)
# - Email notifications on failure
# - Retries with backoff for transient failures
# - Timeout to prevent runaway jobs
# - Parameters with dynamic values
# - Comprehensive tags for organization and cost tracking

try:
    pipelines = w.pipelines.list_pipelines()
    pipeline_list = list(pipelines)
    
    if pipeline_list:
        pipeline_id = pipeline_list[0].pipeline_id
        current_user = w.current_user.me()
        user_email = current_user.user_name
        
        # Create production-ready job
        prod_job = w.jobs.create(
            name="Production: Customer Data Pipeline - Daily",
            
            # Tasks with clear descriptions
            tasks=[
                jobs.Task(
                    task_key="ingest_and_transform",
                    description="Ingest customer data from source systems using SDP pipeline. Applies transformations and data quality checks. Outputs to silver.customers table.",
                    pipeline_task=jobs.PipelineTask(
                        pipeline_id=pipeline_id,
                        full_refresh=False  # Incremental (only new data)
                    ),
                    timeout_seconds=3600,  # 1 hour timeout per task
                )
            ],
            
            # Time-based trigger: Daily at 2 AM UTC (off-peak)
            schedule=jobs.CronSchedule(
                quartz_cron_expression="0 0 2 * * ?",
                timezone_id="UTC",
                pause_status=jobs.PauseStatus.UNPAUSED
            ),
            
            # Email notifications (failure only for production)
            email_notifications=jobs.JobEmailNotifications(
                on_failure=[user_email],
                on_success=[],
                no_alert_for_skipped_runs=True
            ),
            
            # Retry configuration for transient failures
            max_retries=2,
            min_retry_interval_millis=300000,  # 5 minutes between retries
            retry_on_timeout=True,
            
            # Overall job timeout
            timeout_seconds=7200,  # 2 hours max
            
            # Parameters with dynamic values
            parameters=[
                jobs.JobParameterDefinition(
                    name="environment",
                    default="production"
                ),
                jobs.JobParameterDefinition(
                    name="run_date",
                    default="{{job.start_time.iso_date}}"
                )
            ],
            
            # Comprehensive tags for organization and cost tracking
            tags={
                "environment": "production",
                "team": "data-engineering",
                "pipeline": "customer-data",
                "criticality": "high",
                "owner": user_email,
                "cost_center": "analytics",
                "schedule": "daily"
            }
        )
        
        print(f"Production-ready job created!")
        print(f"Job ID: {prod_job.job_id}")
        print(f"\nConfiguration:")
        print(f"   - Schedule: Daily at 2 AM UTC")
        print(f"   - Timeout: 2 hours (job), 1 hour (task)")
        print(f"   - Retries: 2 (with 5 min intervals)")
        print(f"   - Notifications: On failure to {user_email}")
        print(f"   - Tags: {len(prod_job.settings.tags)} tags for organization")
        print(f"\nView job: {w.config.host}/#job/{prod_job.job_id}")
        
    else:
        print("No pipelines found.")
        
except Exception as e:
    print(f"Error: {e}")

### Job Management Operations

**Common operations for managing jobs:**

---

### **Update a Job**

**SDK:**
```python
w.jobs.update(
    job_id=job_id,
    new_settings=jobs.JobSettings(
        name="Updated Job Name",
        schedule=jobs.CronSchedule(
            quartz_cron_expression="0 0 3 * * ?",  # Change to 3 AM
            timezone_id="UTC"
        )
    )
)
```

**CLI:**
```bash
databricks jobs update --job-id 12345 --json @updated-config.json
```

---

### **Pause/Resume Schedule**

```python
# Pause a scheduled job
w.jobs.update(
    job_id=job_id,
    new_settings=jobs.JobSettings(
        schedule=jobs.CronSchedule(
            quartz_cron_expression="0 0 2 * * ?",
            timezone_id="UTC",
            pause_status=jobs.PauseStatus.PAUSED
        )
    )
)

# Resume
w.jobs.update(
    job_id=job_id,
    new_settings=jobs.JobSettings(
        schedule=jobs.CronSchedule(
            quartz_cron_expression="0 0 2 * * ?",
            timezone_id="UTC",
            pause_status=jobs.PauseStatus.UNPAUSED
        )
    )
)
```

---

### **Clone a Job**

Clone via the UI:
1. Go to **Jobs & Pipelines** in the sidebar
2. Click the job name
3. Click the menu (**...**) next to **Run now**
4. Select **Clone job**

---

### **Delete a Job**

```python
w.jobs.delete(job_id=job_id)
print(f"Deleted job {job_id}")
```

---

### **Export Job Configuration**

For version control, use **Declarative Automation Bundles**:

```bash
# Generate bundle configuration from an existing job
databricks bundle generate job --existing-job-id 12345

# This creates YAML files you can check into Git
```

Or export manually via SDK:

```python
job = w.jobs.get(job_id=job_id)

job_config = {
    "name": job.settings.name,
    "tasks": [task.as_dict() for task in job.settings.tasks],
    "schedule": job.settings.schedule.as_dict() if job.settings.schedule else None,
    "tags": job.settings.tags
}

import json
with open(f"job_{job_id}.json", "w") as f:
    json.dump(job_config, f, indent=2)
```

In [0]:
# ============================================
# CLEANUP: Delete Demo Jobs (Optional)
# ============================================

# Uncomment and run this cell to clean up demo jobs

# try:
#     all_jobs = list(w.jobs.list())
#     
#     deleted_count = 0
#     for job in all_jobs:
#         if "Demo:" in job.settings.name or "Challenge" in job.settings.name:
#             print(f"Deleting: {job.settings.name} (ID: {job.job_id})")
#             w.jobs.delete(job_id=job.job_id)
#             deleted_count += 1
#     
#     print(f"\nDeleted {deleted_count} demo jobs")
#     
# except Exception as e:
#     print(f"Error: {e}")

print("Uncomment the code above to delete demo jobs")

## Summary: Lakeflow Jobs Mastery

Congratulations! You've learned how to create and manage Lakeflow Jobs.

---

### **Key Concepts:**

**1. Jobs vs Pipelines:**
* Lakeflow Spark Declarative Pipelines (SDP) = data transformations (the "what")
* Lakeflow Jobs = orchestration (the "when" and "how")
* Jobs orchestrate triggered pipelines on schedules or triggers

**2. Three Core Concepts:**
* **Jobs** — Primary resource for coordinating and scheduling tasks (represented as a DAG)
* **Tasks** — Units of work: pipeline, notebook, Python script, SQL, dbt, If/else, For each
* **Triggers** — When to run: cron schedule, file arrival, continuous, manual

**3. Databricks SDK:**
* `WorkspaceClient()` — Initialize client
* `w.jobs.create()` — Create jobs
* `w.jobs.run_now()` — Trigger runs (with optional parameter overrides)
* `w.jobs.list_runs()` — Monitor execution
* `w.jobs.update()` — Modify jobs
* `w.jobs.delete()` — Remove jobs

**4. Task Types:**
* **Pipeline Task** — Run SDP pipelines (triggered mode only)
* **Notebook Task** — Run notebooks (serverless by default)
* **Python Script Task** — Run Python files
* **SQL Task** — Run SQL queries
* **Control Flow** — If/else branching, For each looping, Run if conditions

**5. Triggers:**
* **Cron** — Time-based (daily, hourly, etc.)
* **File arrival** — Event-driven (Unity Catalog storage locations)
* **Continuous** — Always running
* **Manual** — On-demand via UI, API, or CLI

---

### **Best Practices:**

* Use descriptive naming conventions with environment and frequency
* Tag everything for organization and cost tracking
* Use serverless compute (the default) for most task types
* Configure retries, timeouts, and failure notifications
* Monitor with `system.lakeflow` system tables and dashboards
* Use **Declarative Automation Bundles** for CI/CD and version control
* Make jobs reusable with parameters and dynamic value references
* Document purpose, ownership, and dependencies in task descriptions
* Test in dev/staging targets before deploying to production

---

### **Next Steps:**

1. **Create production jobs:**
   * Identify pipelines to automate
   * Define schedules and SLAs
   * Configure monitoring and alerts

2. **Implement CI/CD with Declarative Automation Bundles:**
   * Store job configs as YAML in Git
   * Define dev/staging/prod targets
   * Automate deployment with `databricks bundle deploy`

3. **Monitor and optimize:**
   * Query `system.lakeflow` tables for job health
   * Join with `system.billing` for cost analysis
   * Build dashboards for operational visibility

---

### **Resources:**

* [Lakeflow Jobs](https://docs.databricks.com/aws/en/jobs/)
* [Lakeflow Spark Declarative Pipelines Concepts](https://docs.databricks.com/aws/en/ldp/concepts/)
* [Lakeflow Pipelines Editor](https://docs.databricks.com/aws/en/ldp/multi-file-editor/)
* [Pipeline Task for Jobs](https://docs.databricks.com/aws/en/jobs/pipeline/)
* [Automate Job Creation and Management](https://docs.databricks.com/aws/en/jobs/automate/)
* [Declarative Automation Bundles](https://docs.databricks.com/aws/en/dev-tools/bundles/)
* [Monitor Job Costs with System Tables](https://docs.databricks.com/aws/en/admin/system-tables/jobs-cost/)
* [Databricks SDK for Python](https://databricks-sdk-py.readthedocs.io/)